# Actividad 1: Exploración de datos de eBird (aves de Esmeraldas, Ecuador)

**Curso:** Optimización de la gestión de ecosistemas marinos y recursos pesqueros en Latinoamérica mediante ciencia abierta (OHW en español, edición 2026)  
**Autor:** Kevin Tapia-Rodriguez  
**Lenguaje:** R

**Objetivos de la actividad:**
1. Cargar un conjunto de datos desde un archivo CSV.
2. Mostrar las primeras filas.
3. Describir brevemente el conjunto de datos.

## Descripción del conjunto de datos

**Contexto:** estos datos forman parte del trabajo del **informe técnico de aves marinas** del Instituto Público de Investigación de Acuicultura y Pesca (**IPIAP**), *Dinámica de aves marinas de la Corriente de Humboldt en respuesta a la disponibilidad de peces pelágicos pequeños durante los eventos El Niño 2023 y 2026 (litoral continental de Ecuador)*. Este repositorio es un resultado alternativo de esa investigación: la herramienta de depuración de los registros de eBird que se usaron en el análisis.

Los datos provienen del **eBird Basic Dataset (EBD)**, la base de observaciones de aves del Cornell Lab of Ornithology, aportadas por observadores de todo el mundo.

- **Origen:** descarga del EBD para la provincia de **Esmeraldas, Ecuador** (código `EC-E`), de enero de 2020 a agosto de 2026 (versión `relAug-2026`).
- **Base completa:** 160.098 registros, 713 especies y 7.669 listas (*checklists*). No se incluye en este repositorio por las políticas de uso de eBird: cada usuario debe solicitarla en [ebird.org/data/download](https://ebird.org/data/download).
- **Muestra usada aquí:** `data/muestra/ebd_muestra_esmeraldas.csv`, con **102 registros** de 6 listas completas. Incluye las cinco especies focales de aves marinas del proyecto, distintos tipos de protocolo (en ruta, por área, incidental) y una lista de un proyecto de censo (*Coastal Shorebird Survey*).
- **Privacidad:** en la muestra se vaciaron las columnas `OBSERVER ID`, `OBSERVER ORCID ID`, `CHECKLIST COMMENTS` y `SPECIES COMMENTS` para proteger a los observadores.

**Cada fila es una especie registrada en una lista.** Columnas clave:

| Columna | Significado |
|---|---|
| `SAMPLING EVENT IDENTIFIER` | Identificador de la lista (evento de muestreo) |
| `COMMON NAME` / `SCIENTIFIC NAME` | Nombre común (inglés) y científico de la especie |
| `CATEGORY` | Nivel taxonómico (`species`, `issf` = subespecie, `spuh`, `slash`…) |
| `OBSERVATION COUNT` | Número de individuos (`X` = presencia sin conteo) |
| `LATITUDE` / `LONGITUDE` | Coordenadas de la localidad |
| `OBSERVATION DATE` | Fecha de la observación |
| `OBSERVATION TYPE` | Protocolo: *Traveling*, *Stationary*, *Area*, *Incidental*… |
| `DURATION MINUTES` / `EFFORT DISTANCE KM` | Esfuerzo de muestreo |
| `ALL SPECIES REPORTED` | 1 = lista completa, 0 = incompleta |
| `PROJECT NAMES` | Proyecto de censo asociado, si existe |

## 1. Cargar paquetes y datos

In [1]:
suppressPackageStartupMessages({
  library(readr)
  library(dplyr)
})

# Ruta relativa: el notebook está en notebooks/ y los datos en data/muestra/
ruta <- "../data/muestra/ebd_muestra_esmeraldas.csv"

# Leemos todo como texto para no perder información (p. ej. conteos "X")
ebd <- read_csv(ruta, col_types = cols(.default = col_character()))

cat("Filas:", nrow(ebd), " | Columnas:", ncol(ebd), "\n")

Filas: 102  | Columnas: 52 


## 2. Primeras filas

El EBD tiene 52 columnas, así que mostramos solo las más informativas.

In [2]:
ebd %>%
  select(`OBSERVATION DATE`, `SAMPLING EVENT IDENTIFIER`, `OBSERVATION TYPE`,
         `COMMON NAME`, `SCIENTIFIC NAME`, `OBSERVATION COUNT`,
         LATITUDE, LONGITUDE, LOCALITY) %>%
  head()

OBSERVATION DATE,SAMPLING EVENT IDENTIFIER,OBSERVATION TYPE,COMMON NAME,SCIENTIFIC NAME,OBSERVATION COUNT,LATITUDE,LONGITUDE,LOCALITY
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
2021-11-01,S104124006,Traveling,American Oystercatcher,Haematopus palliatus,6,0.3866667,-80.0473333,Estuario de Cojimíes-Isla del Amor
2021-11-01,S104124006,Traveling,Black-bellied Plover,Pluvialis squatarola,5,0.3866667,-80.0473333,Estuario de Cojimíes-Isla del Amor
2021-11-01,S104124006,Traveling,American Golden-Plover,Pluvialis dominica,2,0.3866667,-80.0473333,Estuario de Cojimíes-Isla del Amor
2021-11-01,S104124006,Traveling,Semipalmated Plover,Charadrius semipalmatus,200,0.3866667,-80.0473333,Estuario de Cojimíes-Isla del Amor
2021-11-01,S104124006,Traveling,Wilson's Plover,Anarhynchus wilsonia,10,0.3866667,-80.0473333,Estuario de Cojimíes-Isla del Amor
2021-11-01,S104124006,Traveling,Hudsonian Whimbrel,Numenius hudsonicus,10,0.3866667,-80.0473333,Estuario de Cojimíes-Isla del Amor


Nombres de todas las columnas disponibles:

In [3]:
names(ebd)

[1] "GLOBAL UNIQUE IDENTIFIER"   "LAST EDITED DATE"          
 [3] "TAXONOMIC ORDER"            "CATEGORY"                  
 [5] "TAXON CONCEPT ID"           "COMMON NAME"               
 [7] "SCIENTIFIC NAME"            "SUBSPECIES COMMON NAME"    
 [9] "SUBSPECIES SCIENTIFIC NAME" "EXOTIC CODE"               
[11] "OBSERVATION COUNT"          "BREEDING CODE"             
[13] "BREEDING CATEGORY"          "BEHAVIOR CODE"             
[15] "AGE/SEX"                    "COUNTRY"                   
[17] "COUNTRY CODE"               "STATE"                     
[19] "STATE CODE"                 "COUNTY"                    
[21] "COUNTY CODE"                "IBA CODE"                  
[23] "BCR CODE"                   "USFWS CODE"                
[25] "ATLAS BLOCK"                "LOCALITY"                  
[27] "LOCALITY ID"                "LOCALITY TYPE"             
[29] "LATITUDE"                   "LONGITUDE"                 
[31] "OBSERVATION DATE"           "TIME OBSERVATIONS STARTED" 
[33] "OBSERVER ID"                "OBSERVER ORCID ID"         
[35] "SAMPLING EVENT IDENTIFIER"  "OBSERVATION TYPE"          
[37] "PROTOCOL NAME"              "PROTOCOL CODE"             
[39] "PROJECT NAMES"              "PROJECT IDENTIFIERS"       
[41] "DURATION MINUTES"           "EFFORT DISTANCE KM"        
[43] "EFFORT AREA HA"             "NUMBER OBSERVERS"          
[45] "ALL SPECIES REPORTED"       "GROUP IDENTIFIER"          
[47] "HAS MEDIA"                  "APPROVED"                  
[49] "REVIEWED"                   "REASON"                    
[51] "CHECKLIST COMMENTS"         "SPECIES COMMENTS"

## 3. Exploración básica

In [4]:
ebd %>%
  summarise(
    registros   = n(),
    listas      = n_distinct(`SAMPLING EVENT IDENTIFIER`),
    especies    = n_distinct(`SCIENTIFIC NAME`),
    fecha_min   = min(`OBSERVATION DATE`),
    fecha_max   = max(`OBSERVATION DATE`),
    conteos_X   = sum(`OBSERVATION COUNT` == "X")
  )

registros,listas,especies,fecha_min,fecha_max,conteos_X
<int>,<int>,<int>,<chr>,<chr>,<int>
102,6,64,2021-11-01,2026-08-08,3


Registros por lista y protocolo:

In [5]:
ebd %>%
  count(`SAMPLING EVENT IDENTIFIER`, `OBSERVATION DATE`, `OBSERVATION TYPE`,
        `PROJECT NAMES`, name = "registros")

SAMPLING EVENT IDENTIFIER,OBSERVATION DATE,OBSERVATION TYPE,PROJECT NAMES,registros
<chr>,<chr>,<chr>,<chr>,<int>
S104124006,2021-11-01,Traveling,NA,36
S113025897,2022-06-15,Traveling,NA,9
S161604587,2024-01-26,Area,Coastal Shorebird Survey,8
S263890357,2025-07-29,Traveling,NA,14
S365018290,2026-06-28,Traveling,NA,8
S381777123,2026-08-08,Incidental,NA,27


Especies focales de aves marinas presentes en la muestra:

In [6]:
especies_focales <- c("Sula variegata", "Sula nebouxii", "Leucocarbo bougainvilliorum",
                      "Larosterna inca", "Ardenna grisea")

ebd %>%
  filter(`SCIENTIFIC NAME` %in% especies_focales) %>%
  select(`OBSERVATION DATE`, `COMMON NAME`, `SCIENTIFIC NAME`, `OBSERVATION COUNT`, LOCALITY)

OBSERVATION DATE,COMMON NAME,SCIENTIFIC NAME,OBSERVATION COUNT,LOCALITY
<chr>,<chr>,<chr>,<chr>,<chr>
2021-11-01,Blue-footed Booby,Sula nebouxii,1,Estuario de Cojimíes-Isla del Amor
2022-06-15,Blue-footed Booby,Sula nebouxii,200,"Súa, Esmeraldas, EC (0.869, -79.879)"
2025-07-29,Sooty Shearwater,Ardenna grisea,1,Mompiche - area general
2025-07-29,Blue-footed Booby,Sula nebouxii,10,Mompiche - area general
2026-06-28,Inca Tern,Larosterna inca,50,"via quingue - estero del platano, Esmeraldas, EC (0.765, -80.096)"
2026-06-28,Blue-footed Booby,Sula nebouxii,80,"via quingue - estero del platano, Esmeraldas, EC (0.765, -80.096)"
2026-06-28,Guanay Cormorant,Leucocarbo bougainvilliorum,2,"via quingue - estero del platano, Esmeraldas, EC (0.765, -80.096)"
2026-08-08,Blue-footed Booby,Sula nebouxii,1,Same
2026-08-08,Peruvian Booby,Sula variegata,13,Same


## Conclusión

- Se leyó un archivo CSV con `readr::read_csv()` y se exploró con `dplyr`.
- La muestra tiene 102 registros de 6 listas, entre 2021 y 2026, con las cinco especies focales de aves marinas.
- Para trabajar con la base completa, hay que solicitarla en eBird y colocarla en `data/` (ver README).